### Pytorch basic data structure:

Assume an interpretable scenario where a token = vocab word.
So assume a sentence (a prompt) will be passed to the model (we're creating now or any other) so in terms of dimensions we have (1, n)
where 1 is the single prompt, n is the number of tokens in the prompt.

In pytorch this becomes ``torch.tensor([[w1, w2, ..., wn]]) shape: (1, n). dtype: torch.long (integer token ids)``

But we created embedded data so now our tokens are a lot more rich. An individual embedded token wi is really just a T-dimensional
vector, shape (T,) on its own the "1" only comes back once it's sitting inside a prompt/batch again. So once wi is embedded and
placed back into the prompt, in terms of dimensions we have (1, n, T), where T is the dimension we ``embed`` our tokens into,
containing the rich information about the token.

In pytorch this goes from:

``torch.tensor([[w1, w2, ..., wn]]).  shape: (1, n)``

to

``torch.tensor([[[w1], [w2], ..., [wn]]]).   shape: (1, n, 1)``

to

``torch.tensor([[[w11, w12, ..., w1T], [w21, w22, ..., w2T], ..., [wn1, wn2, ..., wnT]]]). shape: (1, n, T)``

The above transitions could be skipped to present the end result, but this is a nice and methodical way to *visualize* how the
extra dimension gets built up. One caveat: this isn't literally what happens under the hood. You never actually manually inflate
each scalar into a size-1 list and then expand it in practice, ``nn.Embedding`` takes your (1, n) LongTensor of integer ids
directly and does a lookup into a weight matrix of shape (vocab_size, T), returning (1, n, T) in a single step. The
(1, n, 1) → (1, n, T) walkthrough above is scaffolding for intuition, not the real mechanism.

When training a model we will be training on many prompts/sentence completions etc, so we convert 1 becomes B in (1, n, T),
producing (B, n, T). (This also assumes every prompt in the batch has the same number of tokens n ``torch.tensor(...)``
needs a rectangular shape, so in practice sequences are padded/truncated to a common length before this step.)

In pytorch we start from ``torch.tensor([[w1, w2, ..., wn]]). shape: (1, n)``

to

``torch.tensor([[w1, w2, ..., wn], ..., [a1, a2, ..., an]]) shape: (B, n)``

to

``torch.tensor([[[w1], [w2], ..., [wn]], ..., [[a1], [a2], ..., [an]]]) shape: (B, n, 1)``

to

``torch.tensor([[[w11, w12, ..., w1T], [w21, w22, ..., w2T], ..., [wn1, wn2, ..., wnT]], ..., [[a11, a12, ..., a1T], [a21, a22, ...., a2T], ..., [an1, an2, ..., anT]]]) shape: (B, n, T)``

## Layer Normalisation

This normalizes each token's vector **independently**, rescaling the embded context to have 0 mean and 1 std in which add learnable parameters so that the model adjust this normalisation. 



In [11]:
import torch
import torch.nn as nn

In [9]:
class LayerNorm(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.ln = nn.LayerNorm(normalized_shape=n_embd)

    def forward(self, x):
        return self.ln(x)

In [19]:
ln = LayerNorm(4)
out = ln(torch.tensor([[[2.0, 4.0, 6.0, 8.0]]]))  # (1, 1, 4)
print(out)   # tensor([[[-1.3416, -0.4472, 0.4472, 1.3416]]], ...)

## This actualy be able to check the parameters weights and biases we create from the module

ln = nn.LayerNorm(4)
print(ln.weight)
print(ln.bias)


tensor([[[-1.3416, -0.4472,  0.4472,  1.3416]]],
       grad_fn=<NativeLayerNormBackward0>)
Parameter containing:
tensor([1., 1., 1., 1.], requires_grad=True)
Parameter containing:
tensor([0., 0., 0., 0.], requires_grad=True)
